In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- Signal Definition & Spectrum Parameters ---
F1 = 1000.0
F2 = 3000.0
F_max = max(F1, F2)

# Frequency axis for FFT (in Hz)
freq_max_plot = 15000.0
N_fft = 4096
frequencies = np.linspace(-freq_max_plot, freq_max_plot, N_fft)

# Continuous signal in time domain for FFT spectrum calculation
T_total = 0.01  # Window duration
t = np.linspace(0, T_total, int(100000))
dt = t[1] - t[0]
x_cont = 3 * np.cos(2 * np.pi * F1 * t) + 4 * np.sin(2 * np.pi * F2 * t)

# Calculation of continuous spectrum X_alpha(f) via FFT
X_cont_fft = np.fft.fftshift(np.fft.fft(x_cont)) * dt
freqs_fft = np.fft.fftshift(np.fft.fftfreq(len(t), d=dt))

@widgets.interact(Fs=widgets.FloatSlider(value=8000.0, min=3000.0, max=14000.0, step=200.0, description='Sampling $F_s$ (Hz):', style={'description_width': 'initial'}, layout=widgets.Layout(width='700px')))
def update_spectrum_plot(Fs):
    print(f"New Sampling Frequency Value: {Fs} Hz")
    
    Ts = 1.0 / Fs
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    plt.subplots_adjust(hspace=0.4, right=0.8)
    
    # 1. Top plot: Periodic spectral replication X_s(f)
    ax_spec = axes[0]
    ax_spec.grid(True, linestyle=':', alpha=0.7)
    ax_spec.set_xlim(-freq_max_plot, freq_max_plot)
    ax_spec.set_ylim(0, 0.035)
    ax_spec.set_xlabel(r'Frequency $F$ (Hz)', fontsize=10)
    ax_spec.set_ylabel(r'$\mathcal{X}_s(f)$', fontsize=10)
    
    # Calculation of periodic spectrum replicas (k from -6 to 6)
    X_sampled_total = np.zeros_like(frequencies, dtype=complex)
    
    k_range = range(-6, 7)
    for k in k_range:
        shift = k * Fs
        X_shifted = np.interp(frequencies - shift, freqs_fft, np.abs(X_cont_fft)) / Ts
        X_sampled_total += X_shifted
        
        color_copy = 'red' if (k != 0 and abs(shift) - F_max < Fs/2) else 'gray'
        ax_spec.plot(frequencies, X_shifted, color=color_copy, alpha=0.3, linestyle='--')

    # Plot total spectrum line
    ax_spec.plot(frequencies, np.abs(X_sampled_total), color='purple', linewidth=1.5, label=r'Total Spectrum $\mathcal{X}_s(f)$')
    
    # Nyquist limit boundary lines
    ax_spec.axvline(Fs/2, color='red', linestyle='--', alpha=0.6, label=r'Nyquist Limits ($\pm F_s/2$)')
    ax_spec.axvline(-Fs/2, color='red', linestyle='--', alpha=0.6)
    
    # Check for aliasing condition (F_s < 2 * F_max)
    has_aliasing = Fs < (2 * F_max)
    if has_aliasing:
        status_text = f"Sampling Frequency Fs = {Fs} Hz | F_max = {F_max} Hz -> WARNING: ALIASING OCCURS (Spectral Overlap)"
        ax_spec.set_facecolor('#fff5f5') 
        ax_spec.set_title(status_text, fontsize=10, fontweight='bold', color='darkred')
    else:
        status_text = f"Sampling Frequency Fs = {Fs} Hz | F_max = {F_max} Hz -> PROPER RECONSTRUCTION (No Overlap)"
        ax_spec.set_facecolor('#f0fff0') 
        ax_spec.set_title(status_text, fontsize=10, fontweight='bold', color='darkgreen')
        
    ax_spec.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

    # 2. Bottom plot: Ideal Filter and Reconstruction
    ax_rec = axes[1]
    ax_rec.grid(True, linestyle=':', alpha=0.7)
    ax_rec.set_xlim(-freq_max_plot, freq_max_plot)
    ax_rec.set_ylim(0, 0.025)
    ax_rec.set_xlabel(r'Frequency $F$ (Hz)', fontsize=10)
    ax_rec.set_ylabel(r'$\mathcal{X}_r(f)$', fontsize=10)
    ax_rec.set_title(r'Ideal Reconstruction Filter $\mathcal{H}_r(f)$ & Reconstructed Spectrum', fontsize=11, fontweight='bold')
    
    cutoff = min(Fs/2, freq_max_plot)
    X_reconstructed = np.where(np.abs(frequencies) <= cutoff, np.interp(frequencies, freqs_fft, np.abs(X_cont_fft)), 0)
    
    color_rec = 'red' if has_aliasing else 'blue'
    ax_rec.plot(frequencies, X_reconstructed, color=color_rec, linewidth=2, label='Reconstructed Signal Spectrum')
    ax_rec.fill_between(frequencies, X_reconstructed, color=color_rec, alpha=0.2)
    
    ax_rec.axvspan(-Fs/2, Fs/2, color='orange', alpha=0.15, label='Ideal Filter Passband (|f| <= Fs/2)')
    ax_rec.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

    plt.show()

    # --- Student Educational Explanation Output ---
    print("\n" + "="*95)
    print(" EDUCATIONAL INTERPRETATION GUIDE FOR STUDENTS:")
    print("="*95)
    print(" 1. TOP PLOT (Sampled Spectrum Xs(f)):")
    print("    - The gray dashed curves represent shifted replicas of the original continuous-time signal spectrum")
    print("      spaced at intervals of the sampling frequency (Fs).")
    print("    - The purple solid line shows the 'Total Spectrum', which is the vertical linear sum of all")
    print("      these overlapping spectral replicas. Where the tails and lobes of adjacent replicas meet,")
    print("      they superimpose, shaping the valleys and peaks of the purple curve.")
    print("    - The red vertical dashed lines mark the Nyquist frequency limits (+- Fs/2).")
    print("    - Individual dashed lobes (gray or red) explicitly show each periodic spectral copy shifted by k*Fs,")
    print("      highlighting how overlapping components contribute to total aliasing.")
    print("")
    print(" 2. BOTTOM PLOT (Ideal Reconstruction Filter & Recovered Spectrum):")
    print("    - The orange shaded band represents the passband of the ideal low-pass reconstruction filter")
    print("      acting strictly within the Nyquist interval (|f| <= Fs/2).")
    print("    - The blue trace shows the resulting spectrum after ideal filtering. If Fs is chosen high enough")
    print("      (satisfying Fs >= 2 * F_max), the baseband replica remains pristine without distortion,")
    print("      enabling flawless signal recovery.")
    print("="*95)